# Model Definition and Evaluation
## Table of Contents
1. [Model Selection](#model-selection)
2. [Feature Engineering](#feature-engineering)
3. [Hyperparameter Tuning](#hyperparameter-tuning)
4. [Implementation](#implementation)
5. [Evaluation Metrics](#evaluation-metrics)
6. [Comparative Analysis](#comparative-analysis)


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np


### What we have done

If you look at the folders in this directory, you see 5 different directories/steps we have taken after creating the baseline model. This is what we have done in each of these directories:

1. **EmbeddingsModel**
    - We build upon the baseline model and created a model that is trained on embeddings of the categorical columns and aggregated numerical features 
2. **Hyperparameter Optimization**
    - We tried to improve the performance of the embedding model by running a hyperparameter optimization using Optuna. However, the best parameters did not improve the results much -> from around 0.16 PR AUC to 0.1727 PR AUC on the validation set.
3. **Feature Engineering**
    - As the hyperparameter optimization did not result in a huge improvement and we probably needed larger changes, we played around with different new numerical features or other ways to combine the categorical features in the embeddings. However, this also did not result in a noticeable improvement. But we noticed that the embeddings were driving factor as the embeddings models PR-AUC ony dropped slightly after removing the numerical input features.
    - During the Literature Review, we found studies that had good results by using tree-based algorithms. Therefore, we tried XGBoost on our existing features. This resulted in a better PR-AUC, especially on the public test set on ChallengeData. It jumped from around 0.13 PR-AUC on the public test set to 0.1895 PR-AUC.
    - We also tried an ensemble approach of combining the embeddings model and XGBoost, however, it seemed as if the embeddings model just impaired XGBoost's performance.
4. **Hyperparameter Optimization: XGBoost**
    - As XGBoost provided better results, we tried running an Optuna study on the XGBoost model to find optimal parameters. This resulted in a slightly better PR-AUC on the validation set, however, on the public test set it performed slightly worse. Possibly, the parameter space was too large in each of our optuna studies, as they did not result in noticeably better performance.
5. **Oversampling/Undersampling**
    - In the literature review, we also found studies that mentioned oversampling/undersampling to counter the class imbalance in fraud classification tasks. Unfortunately, when trying this on our embeddings model, this also did not result in better performance.

This notebook loads the data, does our feature engineering and preprocessing and at the end loads the best embedding and XGBoost model to run inference on the validation set. The PR-AUC values are compared to the baseline model.


## Model Selection

[Discuss the type(s) of models you consider for this task, and justify the selection.]



- ANN using Embeddings for the item, make, goods columns (string based columns) that are combined with aggregated numerical feature
    - Using the embeddings, the model can learn the meaning/relationship of the categorical features to identify typical fraud cases
- Tree based models: XGBoost (based on Literature Review)
    - They are well suited for tabular data

## Feature Engineering

[Describe any additional feature engineering you've performed beyond what was done for the baseline model.]


- For the baseline model, we only used aggregated numerical features. Here we additionally train the models on embeddings of item, make and goods code.

In [81]:
# Load the dataset
train_x_df = pd.read_csv('../data/X_train.csv')
train_y_df = pd.read_csv('../data/Y_train.csv')

test_x_df = pd.read_csv('../data/X_test.csv')

C:\Users\tomhi\AppData\Local\Temp\ipykernel_8396\1067664255.py:2: DtypeWarning: Columns (0: item21, 1: item22, 2: item23, 3: item24, 4: make21, 5: make22, 6: make23, 7: make24, 8: model21, 9: model22, 10: model23, 11: model24, 12: goods_code1, 13: goods_code8, 14: goods_code9, 15: goods_code10, 16: goods_code11, 17: goods_code12, 18: goods_code13, 19: goods_code14, 20: goods_code15, 21: goods_code16, 22: goods_code17, 23: goods_code18, 24: goods_code19, 25: goods_code20, 26: goods_code21, 27: goods_code22, 28: goods_code23, 29: goods_code24) have mixed types. Specify dtype option on import or set low_memory=False.
  train_x_df = pd.read_csv('../data/X_train.csv')
C:\Users\tomhi\AppData\Local\Temp\ipykernel_8396\1067664255.py:5: DtypeWarning: Columns (0: item20, 1: item21, 2: item22, 3: item23, 4: item24, 5: make20, 6: make21, 7: make22, 8: make23, 9: make24, 10: model20, 11: model21, 12: model22, 13: model23, 14: model24, 15: goods_code1, 16: goods_code10, 17: goods_code11, 18: goods_c

### Identify Categorical Columns

In [82]:
item_cols = [f'item{i}' for i in range(1, 25)]
make_cols = [f'make{i}' for i in range(1, 25)]
goods_cols = [f'goods_code{i}' for i in range(1, 25)]

cat_cols = item_cols + make_cols + goods_cols

### Fill NaN values with a flag string

In [83]:
train_x_df[cat_cols] = train_x_df[cat_cols].fillna("NONE")
test_x_df[cat_cols] = test_x_df[cat_cols].fillna("NONE")

### Create Vocabularies

In [84]:
def create_shared_vocab(train_df, test_df, cols):

    all_values = pd.concat([
        train_df[col]
        for col in cols
    ] + [
        test_df[col]
        for col in cols
    ]).astype(str)

    unique_values = sorted(
        set(all_values) - {"NONE"}
    )

    vocab = {
        value: idx + 1
        for idx, value in enumerate(unique_values)
    }

    vocab["NONE"] = 0

    return vocab

item_vocab = create_shared_vocab(
    train_x_df,
    test_x_df,
    item_cols
)

make_vocab = create_shared_vocab(
    train_x_df,
    test_x_df,
    make_cols
)

goods_vocab = create_shared_vocab(
    train_x_df,
    test_x_df,
    goods_cols
)

### Encode Columns based on Vocabulary

In [85]:
def encode_columns(df, cols, vocab):

    for col in cols:

        df[col] = (
            df[col]
            .astype(str)
            .map(vocab)
            .fillna(0)
            .astype(int)
        )

encode_columns(train_x_df,item_cols,item_vocab)
encode_columns(test_x_df,item_cols,item_vocab)
encode_columns(train_x_df,make_cols,make_vocab)
encode_columns(test_x_df,make_cols,make_vocab)
encode_columns(train_x_df,goods_cols,goods_vocab)
encode_columns(test_x_df,goods_cols,goods_vocab)

In [86]:
embedding_sizes = {
    "item": len(item_vocab),
    "make": len(make_vocab),
    "goods": len(goods_vocab)
}

# Identify numerical columns

numeric_cols = [
    col for col in train_x_df.columns
    if col not in cat_cols + ['ID'] + ['Nb_of_items'] + ['model' + str(i) for i in range(1, 25)]
]


train_x_df[numeric_cols] = train_x_df[numeric_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
test_x_df[numeric_cols] = test_x_df[numeric_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

### Create Aggregated Numerical Features

In [87]:
qty_cols = [f'Nbr_of_prod_purchas{i}' for i in range(1, 25)]

price_cols = [f'cash_price{i}' for i in range(1, 25)]

train_x_df['total_item_count'] = train_x_df[qty_cols].sum(axis=1)
train_x_df['total_price'] = train_x_df[price_cols].sum(axis=1)
train_x_df['max_price'] = train_x_df[price_cols].max(axis=1)
train_x_df['mean_price'] = train_x_df[price_cols].sum(axis=1) / train_x_df['total_item_count']

test_x_df['total_item_count'] = test_x_df[qty_cols].sum(axis=1)
test_x_df['total_price'] = test_x_df[price_cols].sum(axis=1)
test_x_df['max_price'] = test_x_df[price_cols].max(axis=1)
test_x_df['mean_price'] = test_x_df[price_cols].sum(axis=1) / test_x_df['total_item_count']

# Refresh numeric cols list to include new features
numeric_cols = [
    col for col in train_x_df.columns
    if col not in cat_cols + ['ID'] + ['Nb_of_items'] + ['model' + str(i) for i in range(1, 25)]
]

C:\Users\tomhi\AppData\Local\Temp\ipykernel_8396\2089262978.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_x_df['total_item_count'] = train_x_df[qty_cols].sum(axis=1)
C:\Users\tomhi\AppData\Local\Temp\ipykernel_8396\2089262978.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_x_df['total_price'] = train_x_df[price_cols].sum(axis=1)
C:\Users\tomhi\AppData\Local\Temp\ipykernel_8396\2089262978.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many time

### Scaling

In [88]:
X_num_log_train_df = train_x_df[numeric_cols].apply(np.log1p)
X_num_log_test_df  = test_x_df[numeric_cols].apply(np.log1p)

### To Tensors

In [89]:
import torch

X_cat = torch.tensor(
    train_x_df[cat_cols].values,
    dtype=torch.long
)

X_cat_test = torch.tensor(
    test_x_df[cat_cols].values,
    dtype=torch.long
)

# Log-transformed numeric tensors -> for the ANN
X_num = torch.tensor(
    X_num_log_train_df.values,
    dtype=torch.float32
)

X_num_test = torch.tensor(
    X_num_log_test_df.values,
    dtype=torch.float32
)

# Raw (untransformed) numeric tensors -> for XGBoost
X_num_raw = torch.tensor(
    train_x_df[numeric_cols].values,
    dtype=torch.float32
)

X_num_test_raw = torch.tensor(
    test_x_df[numeric_cols].values,
    dtype=torch.float32
)

y = train_y_df["fraud_flag"].values

y_tensor = torch.tensor(
    y,
    dtype=torch.float32
)

print(X_cat.shape)
print(X_num.shape)
print(X_num_raw.shape)
print(y_tensor.shape)

torch.Size([92790, 72])
torch.Size([92790, 52])
torch.Size([92790, 52])
torch.Size([92790])


### Split Data

In [90]:
from sklearn.model_selection import train_test_split

train_idx, val_idx = train_test_split(
    np.arange(len(X_cat)), test_size=0.15, random_state=42
)

X_cat_train,  X_cat_val  = X_cat[train_idx],  X_cat[val_idx]
X_num_train,  X_num_val  = X_num[train_idx],  X_num[val_idx]          # log -> ANN
X_num_train_raw, X_num_val_raw = X_num_raw[train_idx], X_num_raw[val_idx]  # raw -> XGBoost
y_train,      y_val      = y_tensor[train_idx], y_tensor[val_idx]

print(type(y_val))
print(np.shape(y_val))

<class 'torch.Tensor'>
torch.Size([13919])


### Dataset Class

In [91]:
from torch.utils.data import Dataset

class FraudDataset(Dataset):

    def __init__(self, X_cat, X_num, y=None):

        self.X_cat = X_cat
        self.X_num = X_num
        self.y = y

    def __len__(self):
        return len(self.X_cat)

    def __getitem__(self, idx):

        if self.y is not None:
            return (
                self.X_cat[idx],
                self.X_num[idx],
                self.y[idx]
            )

        return (
            self.X_cat[idx],
            self.X_num[idx]
        )

### Datasets

In [92]:
train_dataset = FraudDataset(
    X_cat_train,
    X_num_train,
    y_train
)

val_dataset = FraudDataset(
    X_cat_val,
    X_num_val,
    y_val
)

test_dataset = FraudDataset(
    X_cat_test,
    X_num_test
)

## Hyperparameter Tuning

[Discuss any hyperparameter tuning methods you've applied, such as Grid Search or Random Search, and the rationale behind them.]


- We ran Optuna studies for our ANN as well as the XGBoost model
    - It is more efficient (uses fewer trials) as unpromising trials can get pruned
    - We used the TPE Sampler (Tree-structured Parzen Estimator)

You can find the hyperparameter tuning in the notebooks in [1_hyper_param_optim](1_hyper_param_optim/hyperparameter_optim4.ipynb) and [3_hyper_param_optim_xgb](3_hyper_param_optim_xgb/xgboost_hyp_opt.ipynb)

### Optimal Parameters: ANN

```
FraudModel(
  (item_embedding): Embedding(178, 8, padding_idx=0)
  (make_embedding): Embedding(888, 8, padding_idx=0)
  (goods_embedding): Embedding(17029, 103, padding_idx=0)
  (fc): Sequential(
    (0): Linear(in_features=171, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.25, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)
{'lr': 0.00035344399690837556, 'scheduler': 'none', 'make_dim': 8, 'goods_dim': 103}
```

### Optimal Parameters: XGBoost

```
Best params: {'n_estimators': 1266, 'learning_rate': 0.07708419122007407, 'max_depth': 10, 'min_child_weight': 12, 'subsample': 0.8438893019011352, 'colsample_bytree': 0.6518152217337383, 'gamma': 4.2729661681573905}
```

## Implementation

[Implement the final model(s) you've selected based on the above steps.]


### ANN with Embeddings + Numerical Features

In [93]:
import torch.nn as nn

class FraudModel(nn.Module):

    def __init__(
        self,
        embedding_sizes,
        n_numeric,
        item_dim=8,
        make_dim=16,
        goods_dim=64,
        fc_layers=(64, 32),
        dropout_rates=(0.25, 0.2)
    ):
        super().__init__()

        # Shared embedding tables
        self.item_embedding = nn.Embedding(
            embedding_sizes["item"],
            item_dim,
            padding_idx=0
        )
        self.make_embedding = nn.Embedding(
            embedding_sizes["make"],
            make_dim,
            padding_idx=0
        )
        self.goods_embedding = nn.Embedding(
            embedding_sizes["goods"],
            goods_dim,
            padding_idx=0
        )

        total_embedding_dim = (
            item_dim +
            make_dim +
            goods_dim
        )

        # Main classifier
        layers = []
        in_dim = total_embedding_dim + n_numeric

        for out_dim, dropout in zip(fc_layers, dropout_rates):
            layers += [
                nn.Linear(in_dim, out_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_dim = out_dim

        layers.append(nn.Linear(in_dim, 1))
        self.fc = nn.Sequential(*layers)

    # Apply masked mean pooling to filter out padded (empty) embeddings to maximize the importance/influence of actual values
    def masked_mean_pooling(
        self,
        embeddings,
        x
    ):

        # Mask real tokens
        mask = (x != 0).float()

        # Expand mask for embedding dimension
        mask = mask.unsqueeze(-1)

        # Zero-out padded embeddings
        masked_embeddings = embeddings * mask

        # Sum embeddings
        summed = masked_embeddings.sum(dim=1)

        # Count non-padded entries
        counts = mask.sum(dim=1)
        counts = counts.clamp(min=1)

        # Mean pooling
        pooled = summed / counts

        return pooled

    def forward(self, x_cat, x_num):

        # Split categorical groups
        item_x = x_cat[:, 0:24]
        make_x = x_cat[:, 24:48]
        goods_x = x_cat[:, 48:72]

        # Embeddings
        item_emb = self.item_embedding(item_x)
        make_emb = self.make_embedding(make_x)
        goods_emb = self.goods_embedding(goods_x)

        # Masked mean pooling
        item_pooled = self.masked_mean_pooling(
            item_emb,
            item_x
        )
        make_pooled = self.masked_mean_pooling(
            make_emb,
            make_x
        )
        goods_pooled = self.masked_mean_pooling(
            goods_emb,
            goods_x
        )

        # Concatenate pooled embeddings
        x = torch.cat([
            item_pooled,
            make_pooled,
            goods_pooled,
            x_num
        ], dim=1)

        # MLP
        x = self.fc(x)

        return x.squeeze()

### Load Best XGBoost Model

In [94]:
from xgboost import XGBClassifier
import joblib

xgb_model = joblib.load("best_models/xgb_model_opt.pkl")

### Load Best Embedding Model



In [95]:
embeddings_model = FraudModel(
    embedding_sizes=embedding_sizes,
    n_numeric=len(numeric_cols),
    make_dim=8,
    goods_dim=103
)

embeddings_model.load_state_dict(torch.load("best_models/best_final_model_hp2.pt"))

<All keys matched successfully>

### Model Training

Model Training is done in the notebooks in [1_hyper_param_optim](1_hyper_param_optim/hyperparameter_optim4.ipynb) and [3_hyper_param_optim_xgb](3_hyper_param_optim_xgb/xgboost_hyp_opt.ipynb)

### Run Inference on Validation Set

In [96]:
from torch.utils.data import DataLoader
from sklearn.metrics import average_precision_score

BEST_BATCH_SIZE = 512
val_loader   = DataLoader(val_dataset, batch_size=BEST_BATCH_SIZE, shuffle=False)

# Embeddings model predictions
embeddings_model.eval()

with torch.no_grad():
    logits = embeddings_model(X_cat_val,X_num_val)
    embeddings_preds = torch.sigmoid(logits).cpu().numpy()

# XGBoost model predictions
xgb_preds = xgb_model.predict_proba(
    np.hstack([
        X_num_val_raw.cpu().numpy(),
        X_cat_val.cpu().numpy()
    ])
)[:, 1]

## Evaluation Metrics

[Clearly specify which metrics you'll use to evaluate the model performance, and why you've chosen these metrics.]


- PR-AUC: 
    - This metric is very useful for properly evaluating a model’s performance on the minority class in severely imbalanced classification problems
    - The higher the PR-AUC, the better the model is at correctly detecting the minority class, in our case the fraudulent basket samples
    - Specified in the ChallengeData description as the primary metric for comparison

In [97]:
# Calculate average precision for both models
embeddings_ap = average_precision_score(y_val.cpu().numpy(), embeddings_preds)
xgb_ap = average_precision_score(y_val.cpu().numpy(), xgb_preds)

print(f"Embeddings Model: {embeddings_ap:.4f}")
print(f"XGBoost Model: {xgb_ap:.4f}")


Embeddings Model: 0.1727
XGBoost Model: 0.1769


## Comparative Analysis

[Compare the performance of your model(s) against the baseline model. Discuss any improvements or setbacks and the reasons behind them.]


### Baseline Model
```
Sequential(
  (0): Linear(in_features=5, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Dropout(p=0.2, inplace=False)
  (5): Linear(in_features=8, out_features=1, bias=True)
)
```

see [baseline_model.ipynb](../2_BaselineModel/baseline_model.ipynb) for baseline model training

In [98]:
print("++ PR-AUC on Validation Set ++\n")
print("Baseline Model: 0.0669 ")
print(f"Embeddings Model: {embeddings_ap:.4f}")
print(f"XGBoost Model: {xgb_ap:.4f} <- best result")

print("\n\n++ PR-AUC on ChallengeData ++\n")
print("Baseline Model: 0.0623 ")
print("Embeddings Model: 0.1374")
print("XGBoost Model: 0.1895 <- best result")


++ PR-AUC on Validation Set ++

Baseline Model: 0.0669 
Embeddings Model: 0.1727
XGBoost Model: 0.1769 <- best result


++ PR-AUC on ChallengeData ++

Baseline Model: 0.0623 
Embeddings Model: 0.1374
XGBoost Model: 0.1895 <- best result


### Comment

- It is visible that using the embeddings in addition to numerical features that were used for the baseline model siginificantly improved the results. During experimentation, we even noticed that the embeddings play a crucial role in the performance of the ANN. When only using the embeddings, the PR-AUC only dropped a small amount (still > 0.15). 
- When submitting the predictions on a separate testset to ChallengeData, you can see a drop in the PR-AUC for the embeddings model. This could be due to some overfitting on the training data. In contrast, the XGBoost model performed even better on the test set and provided the best result.

Setbacks:
- We experimented with oversampling/undersampling to counter the class imbalance in the dataset. However, this did not improve the results of the model. Possibly, this is due to the PR-AUC metric as this already takes the imbalance into account. 
- The hyperparameter optimization for the embeddings model also barely improved our results. Therefore, without changing the the features and the architecture of the model significantly, we probably won't be able to improve the results further. This is also the reason why we tried XGBoost on our data.
- We also tried an ensemble approach by combining the predictions of the ANN and the XGBoost model but the results did not improve.
